# Universal Tabular ML Pipeline (Classification & Regression)

A **generic, config-driven, production-minded** pipeline. Point it at any tabular file, set the target column, run top-to-bottom.

**What it automatically handles**
- Task detection: binary / multiclass classification or regression
- File formats: CSV, TSV, Excel, Parquet, JSON (or a built-in demo if no file is given)
- Data audit, deterministic cleaning, schema/column-role inference (numeric, low/high-cardinality categorical, dates, IDs, text, constants)
- Correct splitting: **random**, **time-based** (no future leakage) or **group-based** (same entity never in train *and* test)
- Leak-free preprocessing inside one sklearn `Pipeline`
- Baselines → cross-validated model comparison → **one-standard-error model selection** → hyperparameter tuning
- Diagnostics (learning curve), honest test evaluation with bootstrap CIs, threshold selection / prediction intervals
- Explainability (permutation importance, partial dependence), slice analysis, drift monitoring (PSI)
- Persistence of pipeline + model card + importable component module, and a validated `predict()` function

## How to use (3 steps)
1. Edit **only** the `Config` cell (Section 2): `data_path`, `target`, and optionally `id_cols`, `date_cols`, `time_col`, `group_col`.
2. `Run All`.
3. Read the **Run Summary** (Section 19) and check the **Production Checklist** (end).

## Pipeline map
```
Raw data → Audit → Clean → Task detect → Split → Column plan → Leakage guards → EDA (train only)
        → Pipeline[ Cleaner → (Winsorize) → Date features → Category cast → ColumnTransformer → Model ]
        → CV baselines & comparison → 1-SE selection → Tuning → Diagnostics
        → OOF threshold / intervals → Test evaluation (once) → Explain → Slices → Persist → Monitor
```

## Table of contents
1. Setup · 2. Configuration · 3. Pipeline components module · 4. Load data · 5. Audit · 6. Cleaning & task detection · 7. Split · 8. Column plan · 9. Leakage guards · 10. EDA · 11. Pipeline build · 12. Model comparison & selection · 13. Tuning · 14. Diagnostics · 15. Threshold / intervals · 16. Final evaluation · 17. Explainability & slices · 18. Persistence, inference & drift · 19. Run summary · Production checklist

## 1. Setup
Imports are grouped and versions printed — a run is only reproducible if you know the environment.

In [ ]:
import os, sys, json, platform, warnings, importlib, datetime as dt
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn
from scipy.stats import loguniform, randint

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold, TimeSeriesSplit, GroupKFold, StratifiedGroupKFold,
    GroupShuffleSplit, cross_validate, RandomizedSearchCV, learning_curve)
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score, precision_score, recall_score, brier_score_loss,
    log_loss, accuracy_score, balanced_accuracy_score, matthews_corrcoef, confusion_matrix,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    root_mean_squared_error, mean_absolute_error, median_absolute_error, r2_score,
    mean_absolute_percentage_error)
from sklearn.calibration import CalibrationDisplay

try:
    from IPython.display import display
except ImportError:          # allows running as a plain script
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.3})

print(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | sklearn {sklearn.__version__}")

## 2. Configuration — the only cell you normally edit

| Group | Field | Meaning |
|---|---|---|
| Data | `data_path` | Path to your file. `None` → built-in demo (`demo_name`: `breast_cancer` binary, `wine` multiclass, `diabetes` regression) |
| | `target` | Target column name |
| | `id_cols` / `drop_cols` | Identifiers and any columns that must not be used as features (leaky, irrelevant) |
| | `date_cols`, `time_col` | Datetime columns (expanded into month/day-of-week/…); `time_col` enables time-based split |
| | `group_col` | Entity id (e.g. customer) appearing in many rows → group-based split & CV |
| | `positive_class` | Which label counts as "positive" in binary problems |
| Task | `task`, `split` | `auto` (recommended) or force. Auto-split: group → time → random |
| Preprocessing | thresholds | Missing-fraction drop limit, high-cardinality cutoff, text-length cutoff, rare-category grouping |
| Modelling | `selection_rule` | `one_se` (simplest model within 1 SE of best) or `best` |
| Decision | `threshold_strategy` | Binary only: `f1`, `cost` (uses `cost_fn`/`cost_fp`) or `fixed` |

In [ ]:
@dataclass
class Config:
    # ── DATA ──────────────────────────────────────────────────────────────────
    data_path: Optional[str] = None            # e.g. "data/customers.csv"  (None -> demo)
    demo_name: str = "breast_cancer"           # breast_cancer | wine | diabetes
    target: Optional[str] = None               # required when data_path is set
    id_cols: List[str] = field(default_factory=list)
    drop_cols: List[str] = field(default_factory=list)
    date_cols: List[str] = field(default_factory=list)
    time_col: Optional[str] = None
    group_col: Optional[str] = None
    positive_class: Optional[str] = None
    # ── TASK & SPLIT ──────────────────────────────────────────────────────────
    task: str = "auto"                         # auto | binary | multiclass | regression
    split: str = "auto"                        # auto | random | time | group
    test_size: float = 0.20
    cv_folds: int = 5
    seed: int = 42
    # ── PREPROCESSING ────────────────────────────────────────────────────────
    drop_duplicates: bool = True
    max_missing_frac: float = 0.60             # drop features missing more than this (train)
    high_card_threshold: int = 20              # >N unique categories -> target encoding
    text_len_threshold: int = 60               # avg string length above this -> treated as free text (dropped)
    min_category_freq: float = 0.01            # rarer categories are grouped
    scale_numeric: bool = True                 # applied for linear models only
    winsorize_quantiles: Optional[Tuple[float, float]] = None   # e.g. (0.01, 0.99)
    class_weight: Optional[str] = None         # None | "balanced"
    # ── MODELLING ────────────────────────────────────────────────────────────
    selection_rule: str = "one_se"             # one_se | best
    n_search_iter: int = 20
    # ── DECISION LAYER ───────────────────────────────────────────────────────
    threshold_strategy: str = "f1"             # f1 | cost | fixed   (binary only)
    fixed_threshold: float = 0.5
    cost_fn: float = 1.0                       # cost of a false negative
    cost_fp: float = 1.0                       # cost of a false positive
    interval_coverage: float = 0.90            # regression prediction interval
    # ── RUNTIME ──────────────────────────────────────────────────────────────
    max_rows_explain: int = 2000
    n_jobs: int = -1
    artifacts_dir: str = "artifacts"

def validate_config(c: Config) -> None:
    choices = {"task": {"auto", "binary", "multiclass", "regression"},
               "split": {"auto", "random", "time", "group"},
               "selection_rule": {"one_se", "best"},
               "threshold_strategy": {"f1", "cost", "fixed"},
               "class_weight": {None, "balanced"}}
    for k, allowed in choices.items():
        if getattr(c, k) not in allowed:
            raise ValueError(f"Config.{k}={getattr(c, k)!r} invalid; allowed: {sorted(map(str, allowed))}")
    if not 0.05 <= c.test_size <= 0.5: raise ValueError("test_size should be in [0.05, 0.5]")
    if c.cv_folds < 2: raise ValueError("cv_folds must be >= 2")
    if c.data_path and not c.target: raise ValueError("Set Config.target when using your own data")
    if c.split == "time" and not c.time_col: raise ValueError("split='time' requires time_col")
    if c.split == "group" and not c.group_col: raise ValueError("split='group' requires group_col")

CFG = Config()
validate_config(CFG)
np.random.seed(CFG.seed)
print(json.dumps(asdict(CFG), indent=2, default=str))

## 3. Pipeline components module

Custom transformers are written to a real `.py` file (`artifacts/pipeline_components.py`) and imported from there. **Why:** `joblib` pickles classes *by reference*. Classes defined inside a notebook live in `__main__` and the saved model then cannot be loaded in another process. With a module file, anyone can `joblib.load` the model as long as that file is on the path.

In [ ]:
COMPONENTS_SRC = r"""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureCleaner(BaseEstimator, TransformerMixin):
    # Enforces training column order, casts bool->int and replaces +/-inf with NaN.
    def fit(self, X, y=None):
        self.columns_ = list(X.columns)
        return self

    def transform(self, X):
        missing = [c for c in self.columns_ if c not in X.columns]
        if missing:
            raise KeyError(f"Missing required input columns: {missing}")
        X = X[self.columns_].copy()
        for c in X.columns:
            if X[c].dtype == bool:
                X[c] = X[c].astype(int)
        num = X.select_dtypes(include=[np.number]).columns
        if len(num):
            X[num] = X[num].replace([np.inf, -np.inf], np.nan)
        return X


class Winsorizer(BaseEstimator, TransformerMixin):
    # Clips numeric columns to quantiles learned on the TRAINING data only.
    def __init__(self, cols=(), quantiles=(0.01, 0.99)):
        self.cols = cols
        self.quantiles = quantiles

    def fit(self, X, y=None):
        lo, hi = self.quantiles
        self.bounds_ = {}
        for c in self.cols:
            s = pd.to_numeric(X[c], errors="coerce")
            self.bounds_[c] = (float(np.nanquantile(s, lo)), float(np.nanquantile(s, hi)))
        return self

    def transform(self, X):
        X = X.copy()
        for c, (lo, hi) in self.bounds_.items():
            X[c] = X[c].clip(lo, hi)
        return X


class DateFeatures(BaseEstimator, TransformerMixin):
    # Expands datetime columns into numeric parts; the raw column is removed.
    def __init__(self, cols=(), parts=("month", "dayofweek", "day")):
        self.cols = cols
        self.parts = parts

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cols:
            d = pd.to_datetime(X[c], errors="coerce")
            for p in self.parts:
                X[f"{c}__{p}"] = getattr(d.dt, p).astype("float64")
            X = X.drop(columns=c)
        return X


class CategoryCaster(BaseEstimator, TransformerMixin):
    # Makes categorical columns uniformly string-typed (keeps NaN as NaN) so encoders never see mixed types.
    def __init__(self, cols=()):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cols:
            s = X[c]
            X[c] = s.astype(str).where(s.notna(), np.nan)
        return X
"""

ARTIFACTS = Path(CFG.artifacts_dir).resolve()
ARTIFACTS.mkdir(parents=True, exist_ok=True)
(ARTIFACTS / "pipeline_components.py").write_text(COMPONENTS_SRC)

# make the module importable here AND inside joblib worker processes
if str(ARTIFACTS) not in sys.path:
    sys.path.insert(0, str(ARTIFACTS))
os.environ["PYTHONPATH"] = str(ARTIFACTS) + os.pathsep + os.environ.get("PYTHONPATH", "")
importlib.invalidate_caches()
import pipeline_components as pc
print("Components module ready:", [n for n in dir(pc) if n[0].isupper() and n not in {"BaseEstimator", "TransformerMixin"}])

## 4. Load data
`read_any` dispatches on file extension. If no `data_path` is set a demo dataset (with injected missing values, a categorical and an ID column) is used so the notebook always runs.

In [ ]:
def read_any(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"{p} not found (cwd = {Path.cwd()})")
    ext = p.suffix.lower()
    if ext in {".csv", ".txt"}:   return pd.read_csv(p)
    if ext == ".tsv":             return pd.read_csv(p, sep="\t")
    if ext == ".parquet":         return pd.read_parquet(p)
    if ext in {".xlsx", ".xls"}:  return pd.read_excel(p)
    if ext == ".json":            return pd.read_json(p)
    raise ValueError(f"Unsupported file type: {ext}")

def load_demo(name: str, seed: int):
    from sklearn import datasets
    loaders = {"breast_cancer": datasets.load_breast_cancer, "wine": datasets.load_wine, "diabetes": datasets.load_diabetes}
    if name not in loaders:
        raise ValueError(f"demo_name must be one of {list(loaders)}")
    rng = np.random.default_rng(seed)
    bunch = loaders[name](as_frame=True)
    df, target = bunch.frame.copy(), bunch.target.name
    feats = [c for c in df.columns if c != target]
    df["size_group"] = pd.qcut(df[feats[0]], 4, labels=["S", "M", "L", "XL"]).astype(str)   # a categorical feature
    for col in rng.choice(feats, 3, replace=False):                                          # realistic missingness
        df.loc[rng.random(len(df)) < 0.05, col] = np.nan
    df["record_id"] = np.arange(len(df))                                                      # an ID column
    return df, target, ["record_id"]

if CFG.data_path:
    raw = read_any(CFG.data_path)
    TARGET, ID_COLS, SOURCE = CFG.target, list(CFG.id_cols), CFG.data_path
else:
    raw, _demo_target, _demo_ids = load_demo(CFG.demo_name, CFG.seed)
    TARGET, ID_COLS, SOURCE = (CFG.target or _demo_target), (list(CFG.id_cols) or _demo_ids), f"demo:{CFG.demo_name}"

raw.columns = [str(c).strip() for c in raw.columns]
if TARGET not in raw.columns:
    raise KeyError(f"Target '{TARGET}' not in columns: {list(raw.columns)[:30]}")
print(f"Source: {SOURCE} | shape {raw.shape} | target = '{TARGET}'")
display(raw.head())

## 5. Data audit (raw data, before any change)

We document problems *before* fixing them: missing values, duplicates, constant / ID-like / high-cardinality / free-text columns, and target health. Decisions are then made explicitly, not silently.

In [ ]:
def audit(df: pd.DataFrame, target: str) -> pd.DataFrame:
    n = len(df)
    rows = []
    for c in df.columns:
        s = df[c]
        nun = s.nunique(dropna=True)
        flags = []
        if nun <= 1: flags.append("constant")
        if s.isna().mean() > 0.5: flags.append(">50% missing")
        if not pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_datetime64_any_dtype(s):
            if n and nun / n > 0.95: flags.append("id-like")
            if s.dropna().astype(str).str.len().mean() > CFG.text_len_threshold: flags.append("free-text")
            if nun > CFG.high_card_threshold: flags.append("high-cardinality")
        elif pd.api.types.is_integer_dtype(s) and n > 50 and nun / n > 0.99:
            flags.append("id-like (numeric)")
        rows.append({"column": c, "dtype": str(s.dtype), "missing_%": 100 * s.isna().mean(),
                     "n_unique": nun, "flags": ", ".join(flags)})
    return pd.DataFrame(rows).set_index("column")

print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]} | Memory: {raw.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"Exact duplicate rows: {raw.duplicated().sum():,} | Rows with missing target: {raw[TARGET].isna().sum():,}")
num_inf = np.isinf(raw.select_dtypes(include=[np.number])).sum().sum()
print(f"Infinite values: {num_inf}")
display(audit(raw, TARGET))

## 6. Deterministic cleaning & task detection

Only operations that **learn nothing from the data** happen here (so no leakage): infinities→NaN, drop rows without a target, drop duplicates, parse dates, bool→int, remove ID/drop columns. Anything statistical (imputation, scaling, encoding) lives in the pipeline.

**Task detection:** non-numeric or 2-class target → classification; integer-like numeric target with ≤10 distinct values → classification; otherwise regression. Override with `Config.task`.

In [ ]:
DATE_COLS = list(dict.fromkeys(list(CFG.date_cols) + ([CFG.time_col] if CFG.time_col else [])))

def clean_frame(df: pd.DataFrame, cfg: Config, target: str, id_cols, date_cols):
    log, df = [], df.copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    m = df[target].isna()
    if m.any():
        df = df.loc[~m]; log.append(f"dropped {int(m.sum())} rows with missing target")
    for c in date_cols:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    if cfg.time_col:
        m = df[cfg.time_col].isna()
        if m.any():
            df = df.loc[~m]; log.append(f"dropped {int(m.sum())} rows with unparseable time_col")
    if cfg.drop_duplicates:
        subset = [c for c in df.columns if c not in id_cols]
        before = len(df); df = df.drop_duplicates(subset=subset)
        if len(df) < before: log.append(f"dropped {before - len(df)} duplicate rows (ignoring id columns)")
    groups = df[cfg.group_col].copy() if cfg.group_col else None
    to_drop = [c for c in set(id_cols) | set(cfg.drop_cols) | ({cfg.group_col} if cfg.group_col else set()) if c in df.columns]
    if to_drop: df = df.drop(columns=to_drop); log.append(f"removed non-feature columns: {sorted(to_drop)}")
    for c in df.columns:
        if df[c].dtype == bool: df[c] = df[c].astype(int)
    if groups is not None: groups = groups.reset_index(drop=True)
    return df.reset_index(drop=True), groups, log

def detect_task(y: pd.Series, override: str) -> str:
    if override != "auto": return override
    nun = y.nunique()
    if nun < 2: raise ValueError("Target has fewer than 2 distinct values")
    if not pd.api.types.is_numeric_dtype(y) or y.dtype == bool:
        return "binary" if nun == 2 else "multiclass"
    int_like = bool(np.allclose(y, np.round(y)))
    if int_like and nun <= 10: return "binary" if nun == 2 else "multiclass"
    return "regression"

df, GROUPS, CLEAN_LOG = clean_frame(raw, CFG, TARGET, ID_COLS, DATE_COLS)
for line in CLEAN_LOG: print(" •", line)
print(f"Rows after cleaning: {len(df):,}  (from {len(raw):,})")

TASK = detect_task(df[TARGET], CFG.task)
if TASK == "regression":
    y_all = pd.to_numeric(df[TARGET], errors="raise").astype(float)
    CLASS_LABELS = None
else:
    try:    labels = sorted(df[TARGET].unique().tolist())
    except TypeError: labels = sorted(df[TARGET].unique().tolist(), key=str)
    if TASK == "binary" and CFG.positive_class is not None:
        pos = [l for l in labels if str(l) == str(CFG.positive_class)]
        if not pos: raise ValueError(f"positive_class {CFG.positive_class!r} not in {labels}")
        labels = [l for l in labels if l != pos[0]] + pos          # positive class gets index 1
    CLASS_LABELS = labels
    y_all = df[TARGET].map({l: i for i, l in enumerate(labels)}).astype(int)
X_all = df.drop(columns=[TARGET])

PRIMARY = {"binary": "pr_auc", "multiclass": "f1_macro", "regression": "neg_rmse"}[TASK]
PRIMARY_LABEL = {"binary": "PR-AUC", "multiclass": "macro-F1", "regression": "RMSE"}[TASK]
print(f"\nTASK = {TASK} | primary metric = {PRIMARY_LABEL}" + (f" | classes = {CLASS_LABELS}" if CLASS_LABELS else ""))
if TASK == "regression":
    display(y_all.describe().to_frame("target").T)
else:
    dist = y_all.map(dict(enumerate(CLASS_LABELS))).value_counts(normalize=True).rename("share").to_frame()
    display(dist)
    if dist["share"].min() < 0.10: print("⚠ Strong class imbalance: prefer PR-AUC / macro-F1, consider Config.class_weight='balanced' and threshold tuning.")

## 7. Split strategy

The split must mimic **how the model will be used**:
- `random` (stratified for classification) — rows are independent.
- `time` — train on the past, test on the future (prevents look-ahead leakage).
- `group` — an entity (customer, patient, device) appears only on one side.

The test set is then **locked** until Section 16. Cross-validation (matching the split type) on the training part is used for every decision.

In [ ]:
def resolve_split(cfg: Config) -> str:
    if cfg.split != "auto": return cfg.split
    if cfg.group_col: return "group"
    if cfg.time_col: return "time"
    return "random"

SPLIT = resolve_split(CFG)
idx = np.arange(len(X_all))
if SPLIT == "random":
    strat = y_all if (TASK != "regression" and y_all.value_counts().min() >= 2) else None
    tr_idx, te_idx = train_test_split(idx, test_size=CFG.test_size, stratify=strat, random_state=CFG.seed)
elif SPLIT == "group":
    if GROUPS is None: raise ValueError("group split needs Config.group_col")
    tr_idx, te_idx = next(GroupShuffleSplit(1, test_size=CFG.test_size, random_state=CFG.seed).split(X_all, y_all, GROUPS))
    assert set(GROUPS.iloc[tr_idx]).isdisjoint(GROUPS.iloc[te_idx]), "group leakage!"
else:  # time
    order = np.argsort(X_all[CFG.time_col].values, kind="stable")
    cut = int(len(order) * (1 - CFG.test_size))
    tr_idx, te_idx = order[:cut], order[cut:]

X_train, X_test = X_all.iloc[tr_idx].reset_index(drop=True), X_all.iloc[te_idx].reset_index(drop=True)
y_train, y_test = y_all.iloc[tr_idx].reset_index(drop=True), y_all.iloc[te_idx].reset_index(drop=True)
G_TRAIN = GROUPS.iloc[tr_idx].reset_index(drop=True) if GROUPS is not None else None

if TASK != "regression" and SPLIT != "time":
    CV_FOLDS = int(max(2, min(CFG.cv_folds, y_train.value_counts().min())))
else:
    CV_FOLDS = CFG.cv_folds
if CV_FOLDS != CFG.cv_folds: print(f"⚠ cv_folds reduced to {CV_FOLDS} (smallest class too small)")

def build_cv(task, split, folds, seed):
    if split == "time":  return TimeSeriesSplit(n_splits=folds)
    if split == "group": return StratifiedGroupKFold(folds, shuffle=True, random_state=seed) if task != "regression" else GroupKFold(folds)
    return StratifiedKFold(folds, shuffle=True, random_state=seed) if task != "regression" else KFold(folds, shuffle=True, random_state=seed)

CV = build_cv(TASK, SPLIT, CV_FOLDS, CFG.seed)
print(f"Split = {SPLIT} | CV = {type(CV).__name__}({CV_FOLDS}) | train {X_train.shape} | test {X_test.shape}")
if SPLIT == "time":
    t = CFG.time_col
    print(f"Train period: {X_train[t].min()} → {X_train[t].max()}\nTest  period: {X_test[t].min()} → {X_test[t].max()}")
if TASK == "regression":
    print(f"Target mean  train {y_train.mean():.3f} | test {y_test.mean():.3f}")
else:
    print("Class shares  train", y_train.value_counts(normalize=True).round(3).to_dict(), "| test", y_test.value_counts(normalize=True).round(3).to_dict())

## 8. Column plan (feature typing, from the training set only)

Each column gets a **role**; decisions are recorded with reasons so nothing is dropped silently.

| Role | Treatment |
|---|---|
| numeric | median imputation + missing-indicator (+ scaling for linear models) |
| low-cardinality categorical | missing → own category, one-hot (rare categories grouped) |
| high-cardinality categorical | target encoding with internal cross-fitting (leak-safe) |
| date | expanded to month / weekday / day (+ hour, year when meaningful) |
| dropped | constant, too many missing, ID-like, free text |

In [ ]:
def build_column_plan(X: pd.DataFrame, cfg: Config) -> dict:
    plan = {"numeric": [], "low_card": [], "high_card": [], "date": [], "dropped": {}}
    n = len(X)
    for c in X.columns:
        s = X[c]
        if pd.api.types.is_datetime64_any_dtype(s) or c in DATE_COLS:
            plan["date"].append(c); continue
        if s.isna().mean() > cfg.max_missing_frac:
            plan["dropped"][c] = f"{100 * s.isna().mean():.0f}% missing"; continue
        nun = s.nunique(dropna=True)
        if nun <= 1:
            plan["dropped"][c] = "constant"; continue
        if pd.api.types.is_numeric_dtype(s):
            if pd.api.types.is_integer_dtype(s) and n > 50 and nun / n > 0.99:
                plan["dropped"][c] = "id-like numeric"
            else:
                plan["numeric"].append(c)
        else:
            if s.dropna().astype(str).str.len().mean() > cfg.text_len_threshold:
                plan["dropped"][c] = "free text"
            elif nun / n > 0.95:
                plan["dropped"][c] = "id-like"
            elif nun > cfg.high_card_threshold:
                plan["high_card"].append(c)
            else:
                plan["low_card"].append(c)
    return plan

PLAN = build_column_plan(X_train, CFG)
FEATURE_COLS = PLAN["numeric"] + PLAN["low_card"] + PLAN["high_card"] + PLAN["date"]
if not FEATURE_COLS: raise ValueError("No usable feature columns left — check drop_cols / audit output")

# date parts: include 'hour' only if informative, 'year' only when NOT time-splitting (it would extrapolate)
DATE_PARTS = ["month", "dayofweek", "day"]
if any((X_train[c].dt.hour != 0).any() for c in PLAN["date"]): DATE_PARTS.append("hour")
if SPLIT != "time": DATE_PARTS.append("year")
DATE_DERIVED = [f"{c}__{p}" for c in PLAN["date"] for p in DATE_PARTS]

X_train, X_test = X_train[FEATURE_COLS], X_test[FEATURE_COLS]
roles = {**{c: "numeric" for c in PLAN["numeric"]}, **{c: "categorical (one-hot)" for c in PLAN["low_card"]},
         **{c: "categorical (target-enc)" for c in PLAN["high_card"]}, **{c: "date" for c in PLAN["date"]},
         **{c: f"DROPPED: {r}" for c, r in PLAN["dropped"].items()}}
plan_df = pd.DataFrame({"role": pd.Series(roles), "n_unique": X_all[list(roles)].nunique(),
                        "missing_%": 100 * X_train.reindex(columns=list(roles)).isna().mean()})
display(plan_df)
print(f"{len(FEATURE_COLS)} raw features kept, {len(PLAN['dropped'])} dropped")

## 9. Leakage guards

Leakage is the #1 reason models look great in a notebook and fail in production. Three cheap automated smell tests:
1. **Train/test overlap** — identical feature rows on both sides inflate scores.
2. **Near-perfect single feature** — a numeric column almost perfectly rank-correlated with the target usually encodes the answer (e.g. a post-event field).
3. *(later, Section 17)* **Importance dominance** — one feature explaining most of the model.

In [ ]:
# 1) train/test overlap
h_tr = set(pd.util.hash_pandas_object(X_train, index=False))
h_te = pd.util.hash_pandas_object(X_test, index=False)
overlap = float(h_te.isin(h_tr).mean())
print(f"Test rows whose features also appear in train: {100 * overlap:.2f}%" + ("   ⚠ investigate" if overlap > 0.05 else "   ✓"))

# 2) near-perfect single numeric feature
if TASK != "multiclass" and PLAN["numeric"]:
    rho = X_train[PLAN["numeric"]].apply(lambda s: s.corr(y_train, method="spearman")).abs().sort_values(ascending=False)
    display(rho.head(5).to_frame("|spearman| with target"))
    sus = rho[rho > 0.95]
    print("⚠ Suspiciously strong features (possible leakage): " + ", ".join(sus.index) if len(sus) else "No single numeric feature is suspiciously predictive ✓")
else:
    print("Skipped single-feature check (multiclass or no numeric features) — rely on Section 17 importance check.")

## 10. Exploratory data analysis (training data only)
EDA on the full data would let test-set information influence our decisions. Everything below uses `X_train`/`y_train`.

In [ ]:
def grid_axes(n, ncols=3, w=4.6, h=3.2):
    nrows = max(1, int(np.ceil(n / ncols)))
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[n:]: ax.set_visible(False)
    return fig, axes[:n]

def mutual_info_table(X, y, task, seed, max_rows=5000):
    if len(X) > max_rows:
        s = np.random.default_rng(seed).choice(len(X), max_rows, replace=False); X, y = X.iloc[s], y.iloc[s]
    Xe, disc = pd.DataFrame(index=X.index), []
    for c in X.columns:
        if c in PLAN["date"]: continue
        if c in PLAN["numeric"]: Xe[c] = X[c].fillna(X[c].median()); disc.append(False)
        else:                    Xe[c] = X[c].astype(str).factorize()[0]; disc.append(True)
    if Xe.shape[1] == 0: return pd.Series(dtype=float)
    fn = mutual_info_regression if task == "regression" else mutual_info_classif
    return pd.Series(fn(Xe, y, discrete_features=disc, random_state=seed), index=Xe.columns).sort_values(ascending=False)

# --- target distribution
fig, ax = plt.subplots(figsize=(6, 3.5))
if TASK == "regression": ax.hist(y_train, bins=40, color="#4C72B0"); ax.set_xlabel(TARGET)
else:
    y_train.map(dict(enumerate(CLASS_LABELS))).value_counts().sort_index().plot.bar(ax=ax, color="#4C72B0"); ax.set_xlabel("class")
ax.set_title("Target distribution (train)"); plt.tight_layout(); plt.show()

# --- univariate signal ranking (mutual information)
MI = mutual_info_table(X_train, y_train, TASK, CFG.seed)
MI.head(15).iloc[::-1].plot.barh(color="#55A868", figsize=(7, 4.5), title="Univariate signal (mutual information) — top 15")
plt.tight_layout(); plt.show()

In [ ]:
# --- how the strongest numeric features relate to the target
top_num = [c for c in MI.index if c in PLAN["numeric"]][:6]
if top_num:
    fig, axes = grid_axes(len(top_num))
    samp = X_train.sample(min(2000, len(X_train)), random_state=CFG.seed).index
    for ax, c in zip(axes, top_num):
        if TASK == "regression":
            ax.scatter(X_train.loc[samp, c], y_train.loc[samp], s=6, alpha=0.4)
            ax.set_ylabel(TARGET)
        else:
            for k, lab in enumerate(CLASS_LABELS):
                v = X_train.loc[y_train == k, c].dropna()
                ax.hist(v, bins=25, alpha=0.5, density=True, label=str(lab))
            ax.legend(fontsize=7)
        ax.set_title(c, fontsize=9)
    plt.suptitle("Top numeric features vs target", y=1.01); plt.tight_layout(); plt.show()

# --- categorical features vs target
cats = (PLAN["low_card"] + PLAN["high_card"])[:4]
if cats:
    fig, axes = grid_axes(len(cats), ncols=min(4, len(cats)), w=5)
    for ax, c in zip(axes, cats):
        s = X_train[c].astype(str).fillna("NA")
        top = s.value_counts().index[:10]
        if TASK == "multiclass":
            pd.crosstab(s[s.isin(top)], y_train[s.isin(top)], normalize="index").plot.barh(stacked=True, ax=ax, legend=False)
        else:
            tgt = (y_train == 1).astype(float) if TASK == "binary" else y_train
            tgt.groupby(s[s.isin(top)]).mean().sort_values().plot.barh(ax=ax, color="#4C72B0")
            ax.axvline(tgt.mean(), color="red", ls="--", lw=1)
        ax.set_title(f"{c} " + ("(target rate)" if TASK == "binary" else "(mean target)" if TASK == "regression" else "(class mix)"), fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
# --- correlation of numeric features (redundancy / multicollinearity)
nums = PLAN["numeric"][:25]
if len(nums) >= 2:
    corr = X_train[nums].corr(method="spearman")
    fig, ax = plt.subplots(figsize=(min(10, 0.45 * len(nums) + 3),) * 2)
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1); ax.grid(False)
    ax.set_xticks(range(len(nums))); ax.set_xticklabels(nums, rotation=90, fontsize=7)
    ax.set_yticks(range(len(nums))); ax.set_yticklabels(nums, fontsize=7)
    plt.colorbar(im, shrink=0.7); ax.set_title("Spearman correlation"); plt.tight_layout(); plt.show()
    pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack()
    hi = pairs[pairs.abs() > 0.9].sort_values(key=abs, ascending=False)
    print(f"{len(hi)} highly correlated pairs (|rho|>0.9) — fine for trees, may destabilise linear models." if len(hi) else "No pairs with |rho|>0.9 ✓")

# --- skewness (matters for linear models) and missingness
if PLAN["numeric"]:
    skew = X_train[PLAN["numeric"]].skew().abs().sort_values(ascending=False)
    display(skew.head(5).to_frame("|skew|"))
miss = X_train.isna().mean().mul(100)
miss = miss[miss > 0].sort_values(ascending=False)
if len(miss):
    display(miss.to_frame("missing_%"))
    if TASK != "multiclass":
        tgt = (y_train == 1).astype(float) if TASK == "binary" else y_train
        info = {c: (tgt[X_train[c].isna()].mean(), tgt[X_train[c].notna()].mean()) for c in miss.index}
        display(pd.DataFrame(info, index=["target | missing", "target | present"]).T)
        print("Large gaps above mean missingness itself is informative — our pipeline keeps missing-indicator flags for exactly this reason.")
else:
    print("No missing values in the kept features.")

## 11. Pipeline construction

All statistics are learned **inside** the pipeline, so cross-validation refits them on each training fold only. The full chain is a single object — what is trained is exactly what is deployed.

`FeatureCleaner → (Winsorizer) → DateFeatures → CategoryCaster → ColumnTransformer → Model`

In [ ]:
NUMERIC_FOR_CT = PLAN["numeric"] + DATE_DERIVED

def make_preprocessor(scale: bool) -> ColumnTransformer:
    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale and CFG.scale_numeric: num_steps.append(("scale", StandardScaler()))
    blocks = []
    if NUMERIC_FOR_CT: blocks.append(("num", Pipeline(num_steps), NUMERIC_FOR_CT))
    if PLAN["low_card"]:
        blocks.append(("cat_low", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="__missing__")),
            ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=CFG.min_category_freq, sparse_output=False))]),
            PLAN["low_card"]))
    if PLAN["high_card"]:
        blocks.append(("cat_high", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="__missing__")),
            ("te", TargetEncoder(target_type="auto", cv=5, random_state=CFG.seed))]),
            PLAN["high_card"]))
    return ColumnTransformer(blocks, remainder="drop", verbose_feature_names_out=False).set_output(transform="pandas")

def make_pipeline(model, scale: bool) -> Pipeline:
    steps = [("clean", pc.FeatureCleaner())]
    if CFG.winsorize_quantiles and PLAN["numeric"]:
        steps.append(("winsor", pc.Winsorizer(cols=PLAN["numeric"], quantiles=CFG.winsorize_quantiles)))
    steps += [("dates", pc.DateFeatures(cols=PLAN["date"], parts=DATE_PARTS)),
              ("cats", pc.CategoryCaster(cols=PLAN["low_card"] + PLAN["high_card"])),
              ("prep", make_preprocessor(scale)),
              ("model", model)]
    return Pipeline(steps)

# sanity check: what does the model actually see?
_probe = make_pipeline(DummyRegressor() if TASK == "regression" else DummyClassifier(), scale=True).fit(X_train, y_train)
_Z = _probe[:-1].transform(X_train.head(5))
print(f"{len(FEATURE_COLS)} raw features → {_Z.shape[1]} model inputs | any NaN left: {bool(_Z.isna().any().any())}")
display(_Z.head(3))

## 12. Baselines, cross-validated comparison & model selection

- A **dummy baseline** defines the floor: any model must clearly beat it.
- Candidates span the bias/variance spectrum: regularised linear → gradient boosting → random forest.
- **One-standard-error rule:** among models whose score is within 1 SE of the best, choose the *simplest*. Simpler models generalise more reliably, are cheaper, and easier to explain; a 0.001 gain is noise, not signal.

In [ ]:
def get_scoring(task):
    if task == "binary":
        return {"pr_auc": "average_precision", "roc_auc": "roc_auc", "f1": "f1", "neg_brier": "neg_brier_score"}
    if task == "multiclass":
        return {"f1_macro": "f1_macro", "bal_acc": "balanced_accuracy", "roc_auc_ovr": "roc_auc_ovr", "neg_log_loss": "neg_log_loss"}
    return {"neg_rmse": "neg_root_mean_squared_error", "neg_mae": "neg_mean_absolute_error", "r2": "r2"}

SCORING = get_scoring(TASK)
COMPLEXITY = {"Dummy": 0, "Ridge": 1, "LogReg": 1, "HistGradBoost": 2, "RandomForest": 3}

def candidate_models(task, seed, cw):
    if task == "regression":
        return {"Dummy": (DummyRegressor(strategy="mean"), False),
                "Ridge": (Ridge(alpha=1.0), True),
                "RandomForest": (RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=seed), False),
                "HistGradBoost": (HistGradientBoostingRegressor(random_state=seed), False)}
    return {"Dummy": (DummyClassifier(strategy="prior"), False),
            "LogReg": (LogisticRegression(max_iter=3000, class_weight=cw), True),
            "RandomForest": (RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=seed,
                                                    class_weight=("balanced_subsample" if cw else None)), False),
            "HistGradBoost": (HistGradientBoostingClassifier(random_state=seed, class_weight=cw), False)}

CANDIDATES = {n: make_pipeline(m, s) for n, (m, s) in candidate_models(TASK, CFG.seed, CFG.class_weight).items()}

rows, PRIM = [], {}
for name, pipe in CANDIDATES.items():
    res = cross_validate(pipe, X_train, y_train, cv=CV, scoring=SCORING, groups=G_TRAIN, n_jobs=CFG.n_jobs)
    row = {"model": name}
    for k in SCORING:
        v = res[f"test_{k}"]
        sign = -1 if k.startswith("neg_") else 1
        row[(k[4:] if k.startswith("neg_") else k)] = sign * v.mean()
    row["primary_std"] = res[f"test_{PRIMARY}"].std()
    row["fit_s"] = res["fit_time"].mean()
    rows.append(row)
    PRIM[name] = (res[f"test_{PRIMARY}"].mean(), res[f"test_{PRIMARY}"].std(), len(res[f"test_{PRIMARY}"]))

CV_TABLE = pd.DataFrame(rows).set_index("model")
CV_TABLE = CV_TABLE.loc[sorted(PRIM, key=lambda n: -PRIM[n][0])]
display(CV_TABLE)

# --- model selection
real = {n: v for n, v in PRIM.items() if n != "Dummy"}
best_name = max(real, key=lambda n: real[n][0])
b_mean, b_std, b_k = real[best_name]
se = b_std / np.sqrt(b_k)
eligible = [n for n, v in real.items() if v[0] >= b_mean - se]
SELECTED = best_name if CFG.selection_rule == "best" else min(eligible, key=lambda n: (COMPLEXITY[n], -real[n][0]))
print(f"Best by mean: {best_name} | within 1 SE: {eligible} | SELECTED ({CFG.selection_rule}): {SELECTED}")
dummy_gain = PRIM[SELECTED][0] - PRIM["Dummy"][0]
print(f"Gain over dummy baseline on {PRIMARY_LABEL}: {abs(dummy_gain):.4f}" + ("" if dummy_gain > 0 else "   ⚠ model is NOT better than the baseline — check data/target"))

ax = CV_TABLE[CV_TABLE.index != "Dummy"].iloc[::-1].plot.barh(y=CV_TABLE.columns[0], legend=False, figsize=(7, 3), color="#4C72B0")
ax.set_title(f"CV {CV_TABLE.columns[0]} by model (selected: {SELECTED})"); plt.tight_layout(); plt.show()

## 13. Hyperparameter tuning

Randomised search (more efficient than a grid for equal budget) on the **training set only**, with the same CV scheme and the primary metric. We also report the *untuned vs tuned* difference — if tuning gains less than the CV noise, that is useful to know.

In [ ]:
PARAM_DISTS = {
    "LogReg":        {"model__C": loguniform(1e-3, 1e2)},
    "Ridge":         {"model__alpha": loguniform(1e-3, 1e3)},
    "RandomForest":  {"model__n_estimators": randint(200, 600), "model__max_depth": [None, 5, 10, 20],
                      "model__min_samples_leaf": randint(1, 20), "model__max_features": ["sqrt", 0.5, 1.0]},
    "HistGradBoost": {"model__learning_rate": loguniform(0.01, 0.3), "model__max_iter": randint(100, 600),
                      "model__max_leaf_nodes": randint(8, 64), "model__min_samples_leaf": randint(5, 60),
                      "model__l2_regularization": loguniform(1e-3, 10)},
}
search = RandomizedSearchCV(
    CANDIDATES[SELECTED], PARAM_DISTS[SELECTED], n_iter=CFG.n_search_iter, scoring=SCORING[PRIMARY],
    cv=CV, n_jobs=CFG.n_jobs, random_state=CFG.seed, refit=True)
search.fit(X_train, y_train, groups=G_TRAIN)
BEST = search.best_estimator_

sgn = -1 if PRIMARY.startswith("neg_") else 1
print(f"Untuned CV {PRIMARY_LABEL}: {sgn * PRIM[SELECTED][0]:.4f}  →  tuned: {sgn * search.best_score_:.4f}")
print("Best params:", {k.replace("model__", ""): (round(float(v), 5) if isinstance(v, (float, np.floating)) else v) for k, v in search.best_params_.items()})
top = pd.DataFrame(search.cv_results_).sort_values("rank_test_score").head(5)
display(top[["rank_test_score", "mean_test_score", "std_test_score"] + [c for c in top.columns if c.startswith("param_")]])

## 14. Diagnostics — learning curve

Answers: *would more data help, or is the model capacity / feature set the bottleneck?*
- Large train–validation gap → overfitting (more data, regularisation, simpler model).
- Both curves flat and close → under-fitting / limited signal (better features or a richer model).
- Validation still rising at 100% data → collecting more data is likely worthwhile.

In [ ]:
sizes, tr_sc, va_sc = learning_curve(
    clone(BEST), X_train, y_train, cv=CV, groups=G_TRAIN, scoring=SCORING[PRIMARY],
    train_sizes=np.linspace(0.25, 1.0, 4), n_jobs=CFG.n_jobs)
tr_m, va_m = sgn * tr_sc.mean(1), sgn * va_sc.mean(1)
plt.plot(sizes, tr_m, "o-", label="train"); plt.plot(sizes, va_m, "o-", label="validation")
plt.fill_between(sizes, sgn * (va_sc.mean(1) - va_sc.std(1)), sgn * (va_sc.mean(1) + va_sc.std(1)), alpha=0.15)
plt.xlabel("training rows"); plt.ylabel(PRIMARY_LABEL); plt.title("Learning curve"); plt.legend(); plt.tight_layout(); plt.show()
gap = abs(tr_m[-1] - va_m[-1]); improving = abs(va_m[-1] - va_m[-2]) > 0.5 * va_sc.std(1)[-1]
print(f"Train–validation gap at full size: {gap:.4f} | validation still changing noticeably: {bool(improving)}")

## 15. Decision layer — threshold (binary) or prediction interval (regression)

Out-of-fold (OOF) predictions on the training set are used so the decision rule is **never tuned on the test set**.
- **Binary:** the default 0.5 threshold is arbitrary. We choose the threshold maximising F1 or minimising expected cost (`cost_fn`, `cost_fp`) — set the costs to real business values.
- **Regression:** a point estimate is rarely enough; we derive a distribution-free prediction interval from OOF residual quantiles and verify its coverage on the test set.

In [ ]:
def oof_predict(model, X, y, cv, groups, method):
    """Out-of-fold predictions; works for any CV incl. TimeSeriesSplit (uncovered rows -> masked)."""
    out, mask = np.full(len(X), np.nan), np.zeros(len(X), bool)
    for tr, va in cv.split(X, y, groups):
        m = clone(model).fit(X.iloc[tr], y.iloc[tr])
        p = getattr(m, method)(X.iloc[va])
        out[va] = p[:, 1] if method == "predict_proba" else p
        mask[va] = True
    return out, mask

def choose_threshold(y_true, p, strategy, fixed, c_fn, c_fp):
    grid = np.linspace(0.02, 0.98, 97)
    f1s = np.array([f1_score(y_true, p >= t, zero_division=0) for t in grid])
    costs = np.array([c_fn * ((p < t) & (y_true == 1)).sum() + c_fp * ((p >= t) & (y_true == 0)).sum() for t in grid])
    thr = fixed if strategy == "fixed" else float(grid[f1s.argmax()]) if strategy == "f1" else float(grid[costs.argmin()])
    return thr, pd.DataFrame({"threshold": grid, "f1": f1s, "cost": costs})

THRESHOLD, INTERVAL_HALF = None, None
if TASK == "binary":
    oof, mask = oof_predict(BEST, X_train, y_train, CV, G_TRAIN, "predict_proba")
    THRESHOLD, curve = choose_threshold(y_train.to_numpy()[mask], oof[mask], CFG.threshold_strategy,
                                        CFG.fixed_threshold, CFG.cost_fn, CFG.cost_fp)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(curve.threshold, curve.f1); ax[0].set_title("F1 vs threshold (OOF)")
    ax[1].plot(curve.threshold, curve.cost); ax[1].set_title("Expected cost vs threshold (OOF)")
    for a in ax: a.axvline(THRESHOLD, color="red", ls="--", label=f"chosen = {THRESHOLD:.2f}"); a.axvline(0.5, color="grey", ls=":"); a.legend()
    plt.tight_layout(); plt.show()
    print(f"Threshold strategy '{CFG.threshold_strategy}' → {THRESHOLD:.2f} (OOF rows used: {int(mask.sum())})")
elif TASK == "regression":
    oof, mask = oof_predict(BEST, X_train, y_train, CV, G_TRAIN, "predict")
    resid = np.abs(y_train.to_numpy()[mask] - oof[mask])
    INTERVAL_HALF = float(np.quantile(resid, CFG.interval_coverage))
    print(f"{int(100 * CFG.interval_coverage)}% prediction interval half-width (from OOF residuals): ±{INTERVAL_HALF:.4f}")
else:
    print("Multiclass: argmax decision rule (no threshold needed).")

## 16. Final evaluation on the held-out test set (used once)

Reported with **bootstrap 95% confidence intervals** for the primary metric — a single number hides uncertainty. Metrics are task-appropriate; diagnostic plots show *where* the model is wrong, not only how often.

In [ ]:
def predict_all(model, X, thr=None):
    if TASK == "regression":
        return {"pred": model.predict(X), "proba": None}
    proba = model.predict_proba(X)
    if TASK == "binary":
        p1 = proba[:, 1]; return {"pred": (p1 >= thr).astype(int), "proba": p1}
    return {"pred": proba.argmax(1), "proba": proba}

def primary_value(y_true, pred, proba):
    if TASK == "binary":     return average_precision_score(y_true, proba)
    if TASK == "multiclass": return f1_score(y_true, pred, average="macro")
    return root_mean_squared_error(y_true, pred)

def bootstrap_ci(y_true, out, n=300, seed=CFG.seed):
    rng, y_true, vals = np.random.default_rng(seed), np.asarray(y_true), []
    for _ in range(n):
        i = rng.integers(0, len(y_true), len(y_true))
        if TASK != "regression" and len(np.unique(y_true[i])) < len(np.unique(y_true)): continue
        vals.append(primary_value(y_true[i], out["pred"][i], None if out["proba"] is None else out["proba"][i]))
    return np.percentile(vals, [2.5, 97.5])

def compute_metrics(y_true, out):
    y_true, p, pr = np.asarray(y_true), out["pred"], out["proba"]
    if TASK == "regression":
        m = {"RMSE": root_mean_squared_error(y_true, p), "MAE": mean_absolute_error(y_true, p),
             "MedAE": median_absolute_error(y_true, p), "R2": r2_score(y_true, p)}
        if (y_true != 0).all(): m["MAPE"] = mean_absolute_percentage_error(y_true, p)
        m["RMSE of mean-baseline"] = root_mean_squared_error(y_true, np.full_like(y_true, y_train.mean(), dtype=float))
        return m
    if TASK == "binary":
        tn, fp, fn, tp = confusion_matrix(y_true, p).ravel()
        return {"PR-AUC": average_precision_score(y_true, pr), "PR-AUC baseline (prevalence)": y_true.mean(),
                "ROC-AUC": roc_auc_score(y_true, pr), "LogLoss": log_loss(y_true, pr), "Brier": brier_score_loss(y_true, pr),
                "Precision": precision_score(y_true, p, zero_division=0), "Recall": recall_score(y_true, p),
                "Specificity": tn / (tn + fp), "F1": f1_score(y_true, p), "MCC": matthews_corrcoef(y_true, p)}
    m = {"Accuracy": accuracy_score(y_true, p), "Balanced acc": balanced_accuracy_score(y_true, p),
         "Macro-F1": f1_score(y_true, p, average="macro"), "Weighted-F1": f1_score(y_true, p, average="weighted"),
         "LogLoss": log_loss(y_true, pr, labels=list(range(len(CLASS_LABELS))))}
    try: m["ROC-AUC (OvR)"] = roc_auc_score(y_true, pr, multi_class="ovr", labels=list(range(len(CLASS_LABELS))))
    except ValueError: pass
    return m

OUT_TEST = predict_all(BEST, X_test, THRESHOLD)
TEST_METRICS = compute_metrics(y_test, OUT_TEST)
CI = bootstrap_ci(y_test, OUT_TEST)
display(pd.Series(TEST_METRICS, name="test").to_frame())
print(f"{PRIMARY_LABEL} on test = {primary_value(y_test, OUT_TEST['pred'], OUT_TEST['proba']):.4f}  (95% bootstrap CI {CI.round(4)})")
cv_ref, test_val = sgn * search.best_score_, primary_value(y_test, OUT_TEST["pred"], OUT_TEST["proba"])
worse = (test_val > CI[1] if TASK == "regression" else test_val < CI[0]) and abs(test_val - cv_ref) > 0.5 * abs(cv_ref - sgn * PRIM["Dummy"][0]) * 0.1
print(f"CV estimate {cv_ref:.4f} vs test {test_val:.4f}" + (" — ⚠ test is noticeably worse than CV: suspect leakage in CV, overfitting to the search, or distribution shift." if worse else " — consistent ✓"))

In [ ]:
yt, pr_, pb = np.asarray(y_test), OUT_TEST["pred"], OUT_TEST["proba"]
if TASK == "binary":
    fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))
    RocCurveDisplay.from_predictions(yt, pb, ax=ax[0]); ax[0].set_title("ROC")
    PrecisionRecallDisplay.from_predictions(yt, pb, ax=ax[1]); ax[1].axhline(yt.mean(), color="red", ls="--", label="baseline"); ax[1].legend(); ax[1].set_title("Precision–Recall")
    CalibrationDisplay.from_predictions(yt, pb, n_bins=10, ax=ax[2]); ax[2].set_title("Calibration (reliability)")
    ConfusionMatrixDisplay.from_predictions(yt, pr_, display_labels=[str(c) for c in CLASS_LABELS], cmap="Blues", ax=ax[3], colorbar=False)
    ax[3].grid(False); ax[3].set_title(f"Confusion @ threshold {THRESHOLD:.2f}")
    plt.tight_layout(); plt.show()
    print(classification_report(yt, pr_, target_names=[str(c) for c in CLASS_LABELS], zero_division=0))
elif TASK == "multiclass":
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ConfusionMatrixDisplay.from_predictions(yt, pr_, display_labels=[str(c) for c in CLASS_LABELS], normalize="true", cmap="Blues", ax=ax, colorbar=False, values_format=".2f")
    ax.grid(False); ax.set_title("Normalised confusion matrix"); plt.tight_layout(); plt.show()
    print(classification_report(yt, pr_, target_names=[str(c) for c in CLASS_LABELS], zero_division=0))
else:
    res_ = yt - pr_
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    ax[0].scatter(yt, pr_, s=8, alpha=0.5); lim = [min(yt.min(), pr_.min()), max(yt.max(), pr_.max())]
    ax[0].plot(lim, lim, "r--"); ax[0].set_xlabel("actual"); ax[0].set_ylabel("predicted"); ax[0].set_title("Predicted vs actual")
    ax[1].hist(res_, bins=30, color="#4C72B0"); ax[1].set_title("Residuals"); ax[1].axvline(0, color="r")
    ax[2].scatter(pr_, res_, s=8, alpha=0.5); ax[2].axhline(0, color="r"); ax[2].set_xlabel("predicted"); ax[2].set_title("Residuals vs predicted (look for funnels/curves)")
    plt.tight_layout(); plt.show()
    cover = float(np.mean(np.abs(res_) <= INTERVAL_HALF))
    print(f"Empirical coverage of the {int(100 * CFG.interval_coverage)}% interval on test: {100 * cover:.1f}%  (target {100 * CFG.interval_coverage:.0f}%)")

## 17. Explainability & slice analysis

- **Permutation importance** — drop in the primary metric when a raw feature is shuffled (model-agnostic, measured on held-out data, accounts for the whole pipeline).
- **Partial dependence** — average effect of the top features on the prediction.
- **Dominance check** — one feature carrying most of the importance is a leakage / fragility red flag.
- **Slice analysis** — aggregate metrics can hide a segment where the model fails.

In [ ]:
rng = np.random.default_rng(CFG.seed)
sel = rng.choice(len(X_test), min(CFG.max_rows_explain, len(X_test)), replace=False)
Xe, ye = X_test.iloc[sel], y_test.iloc[sel]
perm = permutation_importance(BEST, Xe, ye, scoring=SCORING[PRIMARY], n_repeats=8, random_state=CFG.seed, n_jobs=CFG.n_jobs)
IMP = pd.DataFrame({"feature": X_test.columns, "importance": perm.importances_mean, "std": perm.importances_std}).sort_values("importance", ascending=False)
top = IMP.head(15).iloc[::-1]
plt.barh(top.feature, top.importance, xerr=top["std"], color="#C44E52"); plt.xlabel(f"Δ {PRIMARY_LABEL} when shuffled"); plt.title("Permutation importance (held-out)")
plt.tight_layout(); plt.show()
pos = IMP.importance.clip(lower=0); share = float(pos.iloc[0] / pos.sum()) if pos.sum() > 0 else 0.0
print(f"Top feature '{IMP.feature.iloc[0]}' carries {100 * share:.0f}% of total importance" + ("   ⚠ dominant — verify it is not leaked/fragile" if share > 0.6 else "   ✓"))

if TASK != "multiclass":
    try:
        pdp_feats = [f for f in IMP.feature if f in PLAN["numeric"]][:3]
        if pdp_feats:
            fig, ax = plt.subplots(figsize=(4.6 * len(pdp_feats), 3.6))
            Xp = Xe.copy(); Xp[pdp_feats] = Xp[pdp_feats].astype(float)      # PDP writes float grid values into the column
            PartialDependenceDisplay.from_estimator(BEST, Xp, pdp_feats, kind="average", ax=ax, n_jobs=1)
            plt.tight_layout(); plt.show()
    except Exception as e:
        print("Partial dependence skipped:", type(e).__name__, e)

In [ ]:
def slice_report(cols, min_rows=30, max_cols=3):
    rows = []
    for c in cols[:max_cols]:
        for lvl, idx_ in X_test.groupby(X_test[c].astype(str)).groups.items():
            if len(idx_) < min_rows: continue
            i = np.asarray(idx_)
            try:
                v = primary_value(y_test.iloc[i], OUT_TEST["pred"][i], None if OUT_TEST["proba"] is None else OUT_TEST["proba"][i])
            except ValueError:
                v = np.nan
            rows.append({"feature": c, "level": lvl, "n": len(i), PRIMARY_LABEL: v})
    return pd.DataFrame(rows)

sl = slice_report(PLAN["low_card"])
if len(sl):
    overall = primary_value(y_test, OUT_TEST["pred"], OUT_TEST["proba"])
    display(sl.set_index(["feature", "level"]))
    worst = sl.loc[sl[PRIMARY_LABEL].idxmin() if TASK != "regression" else sl[PRIMARY_LABEL].idxmax()]
    print(f"Overall {PRIMARY_LABEL}: {overall:.4f} | weakest slice: {worst['feature']}={worst['level']} ({worst[PRIMARY_LABEL]:.4f}, n={int(worst['n'])})")
else:
    print("No low-cardinality categorical features with enough rows per level — slice analysis skipped.")

## 18. Persistence, inference & drift monitoring

Artifacts saved in `artifacts/`:

| File | Purpose |
|---|---|
| `model.joblib` | The **entire** fitted pipeline (cleaning → features → model) |
| `pipeline_components.py` | Custom transformer code (must be importable to load the model) |
| `model_card.json` | Task, schema, class labels, threshold/interval, metrics, params, library versions, data fingerprint |
| `reference_sample.csv` | Training sample used as the reference for drift detection |
| `cv_results.csv`, `importance.csv`, `config.json` | Audit trail |

`predict()` validates the schema, applies the same threshold/interval, and returns original class labels. A **round-trip test** proves the reloaded model reproduces in-memory predictions.

In [ ]:
REQUIRED_COLUMNS = list(FEATURE_COLS)

joblib.dump(BEST, ARTIFACTS / "model.joblib")
CV_TABLE.to_csv(ARTIFACTS / "cv_results.csv"); IMP.to_csv(ARTIFACTS / "importance.csv", index=False)
(ARTIFACTS / "config.json").write_text(json.dumps(asdict(CFG), indent=2, default=str))
X_train.sample(min(5000, len(X_train)), random_state=CFG.seed).to_csv(ARTIFACTS / "reference_sample.csv", index=False)

MODEL_CARD = {
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "source": SOURCE, "task": TASK, "target": TARGET, "class_labels": CLASS_LABELS,
    "model": SELECTED, "split": SPLIT, "selection_rule": CFG.selection_rule,
    "required_columns": REQUIRED_COLUMNS, "date_columns": PLAN["date"],
    "column_plan": {k: v for k, v in PLAN.items() if k != "dropped"}, "dropped_columns": PLAN["dropped"],
    "threshold": THRESHOLD, "interval_half_width": INTERVAL_HALF, "interval_coverage": CFG.interval_coverage if TASK == "regression" else None,
    "primary_metric": PRIMARY_LABEL, "cv_primary": round(float(sgn * search.best_score_), 4),
    "test_metrics": {k: round(float(v), 4) for k, v in TEST_METRICS.items()}, "test_primary_ci95": [round(float(x), 4) for x in CI],
    "best_params": {k: (v if isinstance(v, (int, float, str, type(None))) else str(v)) for k, v in search.best_params_.items()},
    "rows": {"train": len(X_train), "test": len(X_test)}, "seed": CFG.seed,
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}
(ARTIFACTS / "model_card.json").write_text(json.dumps(MODEL_CARD, indent=2, default=str))
print("Saved:", sorted(p.name for p in ARTIFACTS.iterdir()))

In [ ]:
def load_bundle(artifacts_dir=ARTIFACTS):
    artifacts_dir = Path(artifacts_dir).resolve()
    if str(artifacts_dir) not in sys.path: sys.path.insert(0, str(artifacts_dir))      # makes pipeline_components importable
    return joblib.load(artifacts_dir / "model.joblib"), json.loads((artifacts_dir / "model_card.json").read_text())

def predict(records: pd.DataFrame, artifacts_dir=ARTIFACTS) -> pd.DataFrame:
    """Score new rows. Extra columns are ignored; required ones are validated."""
    model, card = load_bundle(artifacts_dir)
    missing = sorted(set(card["required_columns"]) - set(records.columns))
    if missing: raise ValueError(f"Missing required columns: {missing}")
    X = records[card["required_columns"]]
    out = pd.DataFrame(index=records.index)
    if card["task"] == "regression":
        p = model.predict(X); h = card["interval_half_width"]
        out["prediction"], out["lower"], out["upper"] = p, p - h, p + h
    else:
        labels, proba = card["class_labels"], model.predict_proba(X)
        if card["task"] == "binary":
            out["probability_" + str(labels[1])] = proba[:, 1]
            out["prediction"] = [labels[int(v)] for v in (proba[:, 1] >= card["threshold"])]
        else:
            for i, l in enumerate(labels): out[f"probability_{l}"] = proba[:, i]
            out["prediction"] = [labels[i] for i in proba.argmax(1)]
    return out

# Round-trip test: reloaded model must reproduce in-memory predictions
chk = predict(X_test.head(100))
ref = predict_all(BEST, X_test.head(100), THRESHOLD)
if TASK == "regression":   assert np.allclose(chk["prediction"], ref["pred"])
elif TASK == "binary":     assert np.allclose(chk.iloc[:, 0], ref["proba"])
else:                      assert [CLASS_LABELS[i] for i in ref["pred"]] == list(chk["prediction"])
print("Round-trip check passed ✓")
display(chk.head())

In [ ]:
# ---- Drift monitoring (Population Stability Index) -------------------------------------------------
def psi(expected, actual, bins=10):
    e, a = pd.Series(expected).dropna(), pd.Series(actual).dropna()
    bins = int(np.clip(len(a) // 20, 3, bins))          # fewer bins for small samples (PSI is biased upward otherwise)
    if len(e) < bins or len(a) < 30 or e.nunique() < 3: return np.nan
    edges = np.unique(np.quantile(e, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3: return np.nan
    edges[0], edges[-1] = -np.inf, np.inf
    pe = np.clip(np.histogram(e, edges)[0] / len(e), 1e-4, None)
    pa = np.clip(np.histogram(a, edges)[0] / len(a), 1e-4, None)
    return float(np.sum((pa - pe) * np.log(pa / pe)))

def drift_report(X_new: pd.DataFrame, reference: pd.DataFrame = None) -> pd.DataFrame:
    ref = reference if reference is not None else pd.read_csv(ARTIFACTS / "reference_sample.csv")
    rows = [{"feature": c, "PSI": psi(ref[c], X_new[c]), "missing_%_new": 100 * X_new[c].isna().mean(),
             "missing_%_ref": 100 * ref[c].isna().mean()} for c in PLAN["numeric"] if c in X_new.columns]
    r = pd.DataFrame(rows).set_index("feature")
    r["status"] = pd.cut(r["PSI"], [-np.inf, 0.1, 0.25, np.inf], labels=["stable", "moderate drift", "SIGNIFICANT drift"])
    return r.sort_values("PSI", ascending=False)

print("Drift: held-out test vs training (should be stable; PSI is noisy on small samples)"); display(drift_report(X_test, X_train).head(5))
if PLAN["numeric"]:
    shifted = X_test.copy(); c0 = IMP[IMP.feature.isin(PLAN["numeric"])].feature.iloc[0]
    shifted[c0] = shifted[c0] + 1.5 * X_train[c0].std()
    print(f"Self-test: artificially shifting '{c0}' by +1.5 std must be flagged"); display(drift_report(shifted, X_train).head(3))

## 19. Run summary

In [ ]:
lines = [
    f"Source        : {SOURCE}   ({len(raw):,} raw rows → {len(df):,} after cleaning)",
    f"Task          : {TASK}" + (f"   classes={CLASS_LABELS}" if CLASS_LABELS else ""),
    f"Split         : {SPLIT}  (train {len(X_train):,} / test {len(X_test):,}), CV = {type(CV).__name__}({CV_FOLDS})",
    f"Features      : {len(FEATURE_COLS)} used, {len(PLAN['dropped'])} dropped {list(PLAN['dropped'])[:5]}",
    f"Baseline      : {PRIMARY_LABEL} = {sgn * PRIM['Dummy'][0]:.4f}",
    f"Selected model: {SELECTED}  (rule: {CFG.selection_rule})",
    f"CV {PRIMARY_LABEL:<9}: {sgn * search.best_score_:.4f}",
    f"Test {PRIMARY_LABEL:<7}: {primary_value(y_test, OUT_TEST['pred'], OUT_TEST['proba']):.4f}   95% CI [{CI[0]:.4f}, {CI[1]:.4f}]",
]
if THRESHOLD is not None:       lines.append(f"Threshold     : {THRESHOLD:.2f}  (strategy: {CFG.threshold_strategy})")
if INTERVAL_HALF is not None:   lines.append(f"Interval      : ±{INTERVAL_HALF:.4f} for {int(100 * CFG.interval_coverage)}% coverage")
lines.append(f"Top features  : {', '.join(IMP.feature.head(5))}")
lines.append(f"Artifacts     : {ARTIFACTS}")
print("\n".join(lines))

---
## Production checklist

**Before trusting the result**
- [ ] Target definition is exactly what the business wants; no feature is recorded *after* the target event (check timestamps, not only Section 9's automatic tests).
- [ ] Split type matches deployment (time-based if the model predicts the future; group-based if it scores new entities).
- [ ] Test metric is close to CV and its CI is acceptable; model clearly beats the dummy baseline.
- [ ] Weakest slice (Section 17) is acceptable; fairness requirements reviewed if people are affected.
- [ ] Costs in `Config` (`cost_fn`, `cost_fp`) are real, finance-approved numbers.

**Before deploying**
- [ ] Move `pipeline_components.py` into your package; pin library versions from `model_card.json`.
- [ ] Wrap `predict()` in an API / batch job with input schema validation and logging of inputs + outputs.
- [ ] Schedule `drift_report()` (PSI > 0.25 → investigate) and track the primary metric once labels arrive.
- [ ] Define a retraining trigger and a rollback model.
- [ ] For decisions with an intervention (campaigns, treatments), A/B test to measure *causal* uplift — prediction accuracy ≠ business impact.

**Common pitfalls this notebook already guards against:** fitting preprocessing on all data · tuning on the test set · random split on temporal/grouped data · ignoring class imbalance · default 0.5 threshold · reporting a single number with no uncertainty · model that cannot be reloaded because its classes lived in `__main__`.

**Easy extensions:** add models (LightGBM/XGBoost) in `candidate_models` + `PARAM_DISTS`; add text columns via TF-IDF in `make_preprocessor`; add SHAP for per-row explanations; add monotonic constraints or calibration (`CalibratedClassifierCV`) for regulated settings.